In [1]:
import os
import shutil
import subprocess
from typing import Any

from dotenv import load_dotenv
from IPython.display import Markdown, display
from utils.agent_visualizer import (
    display_agent_response,
    print_activity,
    reset_activity_context,
    visualize_conversation,
)

from claude_agent_sdk import ClaudeAgentOptions, ClaudeSDKClient

# Load ANTHROPIC_API_KEY and GITHUB_TOKEN from .env once for the whole notebook.
load_dotenv()

# Defining the model name as a constant makes it easy to swap models in one place.
MODEL = "claude-opus-4-5"

# 02 - The Observability Agent

Your engineering team's GitHub Actions workflows fail for dozens of reasons: flaky tests, dependency conflicts, infrastructure hiccups, or real bugs. Manually triaging which failures need immediate attention versus which can wait wastes hours of senior engineer time every week.

An AI observability agent can monitor your CI/CD pipelines 24/7, distinguish signal from noise, and surface only what matters. In this notebook we connect Claude to external systems through the [Model Context Protocol (MCP)](https://modelcontextprotocol.io/docs/getting-started/intro) and build exactly that: a read-only agent that analyzes GitHub Actions health and produces on-call-ready summaries.

**By the end of this cookbook, you'll be able to:**
- Configure stdio-based MCP servers (Git and GitHub) for a `ClaudeSDKClient` agent
- Restrict an agent to a specific tool surface using `allowed_tools` and `disallowed_tools`
- Build a reusable observability agent that triages CI failures across any repo you have access to
- Extend the same pattern to other MCP servers (Slack, PagerDuty, internal APIs) for DevOps automation

This pattern generalizes to any domain where an agent needs live, authenticated access to an external system.

**Need more details on MCP?** See the [Claude Code MCP documentation](https://docs.claude.com/en/docs/claude-code/mcp) for configuration and troubleshooting guidance.


## Prerequisites

**Required Knowledge**
- Python `async`/`await` fundamentals
- Basic familiarity with the Claude Agent SDK (see [Notebook 00](./00_The_one_liner_research_agent.ipynb))
- Awareness of what MCP is — see the [MCP intro](https://modelcontextprotocol.io/docs/getting-started/intro)

**Required Tools**
- Python 3.11+
- An Anthropic API key in `.env` as `ANTHROPIC_API_KEY` ([get one](https://console.anthropic.com))
- A GitHub fine-grained Personal Access Token in `.env` as `GITHUB_TOKEN` ([create one](https://github.com/settings/personal-access-tokens/new)). The default read-only public-repo scope is enough for this notebook.
- Docker running locally (for the GitHub MCP server). Verify with `docker --version`.

**Setup**

Dependencies are installed via `uv sync` from the repo root — this includes `mcp-server-git`, used in the first example. The next cell imports the SDK, loads your `.env`, and defines a single `MODEL` constant we'll reuse throughout the notebook.


## Introduction to the MCP Server
### 1. The Git MCP server

Let's first give our agent the ability to understand and work with Git repositories. By adding the [Git MCP server](https://github.com/modelcontextprotocol/servers/tree/main/src/git) to our agent, it gains access to 13 Git-specific tools that let it examine commit history, check file changes, create branches, and even make commits. This transforms our agent from a passive observer into an active participant in your development workflow.

Two SDK options do the heavy lifting in the cell below:

- **`allowed_tools=["mcp__git"]`** — the `mcp__<server>` prefix is a wildcard that permits every tool exposed by the named MCP server. If you wanted to restrict the agent to just `git_log` and `git_status`, you'd write `["mcp__git__git_log", "mcp__git__git_status"]`.
- **`disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"]`** — without this, the agent could satisfy a "check git status" request by shelling out through `Bash` instead of going through MCP. Disallowing non-MCP tools is what makes the restriction meaningful.


In [2]:
# Get the git repository root (mcp_server_git requires a valid git repo path)
# os.getcwd() may return a subdirectory, so we find the actual repo root
git_executable = shutil.which("git")
if git_executable is None:
    raise RuntimeError("Git executable not found in PATH")

git_repo_root = subprocess.run(  # noqa: S603
    [git_executable, "rev-parse", "--show-toplevel"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

# Define our git MCP server (installed via uv sync from pyproject.toml)
git_mcp: dict[str, Any] = {
    "git": {
        "command": "uv",
        "args": ["run", "python", "-m", "mcp_server_git", "--repository", git_repo_root],
    }
}

In [3]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        mcp_servers=git_mcp,
        allowed_tools=["mcp__git"],
        # disallowed_tools ensures the agent ONLY uses MCP tools, not Bash with git commands
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(
        "Explore this repo's git history and provide a brief summary of recent activity."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__git__git_log()
🤖 Using: mcp__git__git_status()
🤖 Using: mcp__git__git_branch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__git__git_log()
🤖 Using: mcp__git__git_status()
🤖 Using: mcp__git__git_branch()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


In [4]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
## Git Repository Summary

### Current Branch
You're on the **`upstream-contribution`** branch (up to date with origin), with `main` also available locally.

---

### Recent Commit Activity (Last ~5 Days)

| Date | Author | Summary |
|------|--------|---------|
| **Nov 27, 2025** | costiash | 3 commits enhancing the **Claude Agent SDK** - improved chief of staff agent, notebooks, observability agent, research agent, documentation, and utilities |
| **Nov 26, 2025** | Pedram Navid | Added GitHub issue templates, `/review-issue` command, `/add-registry` slash command, and new cookbook entries |
| **Nov 25, 2025** | Elie Schoppik | Renamed PTC notebook to `programmatic_tool_calling_ptc.ipynb` for clarity |
| **Nov 24, 2025** | henrykeetay | Added **tool search cookbook** |
| **Nov 24, 2025** | Alex Notov | Multiple merges consolidating cookbooks for Opus 4.5, dependency updates |
| **Nov 23, 2025** | Cal Rueb | Simplified crop tool notebook with Claude Agent SDK section |
| **Nov 23, 2025** | Pedram Navid | PR comment fixes and lint cleanup |

---

### Key Themes in Recent Development
1. **Claude Agent SDK enhancements** - Major work on agent implementations (research, chief of staff, observability agents)
2. **New cookbooks** - Tool search, crop tool, programmatic tool calling
3. **CI/CD improvements** - PR review workflows, issue templates, slash commands
4. **Documentation** - Added troubleshooting guides, codebase overviews

---

### Working Directory Status
There are **uncommitted changes** in your working directory:
- **22 modified files** (mostly in `claude_agent_sdk/`)
- **4 deleted files** (documentation files in `docs/`)
- **6 untracked files** (new reports, plans, VS Code config)

These changes appear to be further work on the Claude Agent SDK agents, notebooks, and utilities that haven't been staged or committed yet.

Notice that the agent picked `git_log`, `git_status`, and `git_branch` on its own — we told it *what* to investigate, not *how*. Because `disallowed_tools` blocks `Bash`, every git interaction had to go through MCP rather than shelling out. Swap the prompt to "summarize the last 10 commits touching `src/`" or "what branches are ahead of `main`?" and you'll see the same mechanism do the routing.


### 2. The GitHub MCP server

Now let's level up from local Git operations to full GitHub platform integration. By switching to the [official GitHub MCP server](https://github.com/github/github-mcp-server/tree/main), our agent gains access to over 100 tools that interact with GitHub's entire ecosystem – from managing issues and pull requests to monitoring CI/CD workflows and analyzing code security alerts. This server can work with both public and private repositories, giving your agent the ability to automate complex GitHub workflows that would typically require multiple manual steps.

#### Step 1: Set up your GitHub Token

You need a GitHub Personal Access Token. Get one [here](https://github.com/settings/personal-access-tokens/new) and put in the .env file as ```GITHUB_TOKEN="<token>"```
> Note: When getting your token, select "Fine-grained" token with the default options (i.e., public repos, no account permissions), that'll be the easiest way to get this demo working.

Also, for this example you will have to have [Docker](https://www.docker.com/products/docker-desktop/) running on your machine. Docker is required because the GitHub MCP server runs in a containerized environment for security and isolation.

**Docker Quick Setup:**
- Install Docker Desktop from [docker.com](https://www.docker.com/products/docker-desktop/)
- Ensure Docker is running (you'll see the Docker icon in your system tray)
- Verify with `docker --version` in your terminal
- **Troubleshooting:** If Docker won't start, check that virtualization is enabled in your BIOS. For detailed setup instructions, see the [Docker documentation](https://docs.docker.com/get-docker/)

#### Step 2: Define the mcp server and start the agent loop!

In [5]:
# Define our GitHub MCP server. The Docker container reads the token from
# GITHUB_PERSONAL_ACCESS_TOKEN, which we populate from the GITHUB_TOKEN env var.
github_mcp: dict[str, Any] = {
    "github": {
        "command": "docker",
        "args": [
            "run",
            "-i",
            "--rm",
            "-e",
            "GITHUB_PERSONAL_ACCESS_TOKEN",
            "ghcr.io/github/github-mcp-server",
        ],
        "env": {"GITHUB_PERSONAL_ACCESS_TOKEN": os.environ.get("GITHUB_TOKEN")},
    }
}

In [6]:
# run our agent
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        # disallowed_tools ensures the agent ONLY uses MCP tools, not Bash with gh CLI
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(
        "Search for the anthropics/claude-agent-sdk-python repository and give me a few key facts about it."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__github__search_repositories()
✓ Tool completed
🤖 Thinking...


In [7]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
Here are the key facts about the **anthropics/claude-agent-sdk-python** repository:

| Fact | Details |
|------|---------|
| **Full Name** | anthropics/claude-agent-sdk-python |
| **URL** | https://github.com/anthropics/claude-agent-sdk-python |
| **Language** | Python |
| **Stars** | ⭐ 3,357 |
| **Forks** | 🍴 435 |
| **Open Issues** | 149 |
| **Created** | June 11, 2025 |
| **Last Updated** | December 4, 2025 |
| **Default Branch** | main |
| **Visibility** | Public |
| **Archived** | No |

This is the official Python SDK for building Claude agents, maintained by Anthropic. It's quite popular with over 3,300 stars and has an active community with 435 forks. The repository is actively maintained (recently updated) and has a notable number of open issues (149), which suggests active development and community engagement.

The agent reached the GitHub API through a Docker-isolated MCP server rather than the `gh` CLI or raw HTTP requests. One practical consequence: each query pays a Docker cold-start cost. For high-frequency use you'd run the MCP server as a long-lived process instead of spawning a container per call.


Here's the shape of the system we're building. The agent stays read-only; the MCP server and its Docker container are the only components that talk to GitHub.

```mermaid
sequenceDiagram
    participant User
    participant Agent as ClaudeSDKClient
    participant MCP as GitHub MCP (Docker)
    participant API as GitHub API

    User->>Agent: "Analyze CI health for facebook/react"
    Agent->>MCP: mcp__github__list_commits / get_commit / pull_request_read
    MCP->>API: authenticated requests (GITHUB_TOKEN)
    API-->>MCP: JSON responses
    MCP-->>Agent: structured tool results
    Agent-->>User: on-call-ready triage summary
```


## Real use case: An observability agent

Now, with such simple setup we can already have an agent acting as self-healing software system!

In [8]:
prompt = """Analyze the CI health for facebook/react repository.

Examine the most recent runs of the 'CI' workflow and provide:
1. Current status and what triggered the run (push, PR, schedule, etc.)
2. If failing: identify the specific failing jobs/tests and assess severity
3. If passing: note any concerning patterns (long duration, flaky history)
4. Recommended actions with priority (critical/high/medium/low)

Provide a concise operational summary suitable for an on-call engineer.
Do not create issues or PRs - this is a read-only analysis."""

# Reuse the github_mcp config defined earlier rather than redefining it.
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        model=MODEL,
        mcp_servers=github_mcp,
        allowed_tools=["mcp__github"],
        # IMPORTANT: disallowed_tools is required to actually RESTRICT tool usage.
        # Without this, allowed_tools only controls permission prompting, not availability.
        # The agent would still have access to Bash (and could use `gh` CLI instead of MCP).
        disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"],
        permission_mode="acceptEdits",
    )
) as agent:
    await agent.query(prompt)
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__list_commits()
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__get_file_contents()
🤖 Using: mcp__github__list_pull_requests()
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__sea

In [9]:
display(Markdown(f"\nResult:\n{messages[-1].result}"))


Result:
Based on my comprehensive analysis of the facebook/react repository CI infrastructure, here is the operational summary:

---

# CI Health Analysis: facebook/react

## Executive Summary
**Overall Status: 🟢 HEALTHY**

The React repository's CI appears to be in good health. Recent commits to `main` have been successfully merged, and active PRs show passing CodeSandbox builds.

---

## 1. CI Infrastructure Overview

### Primary Workflows
| Workflow | Trigger | Purpose |
|----------|---------|---------|
| `runtime_build_and_test.yml` | Push to main, PRs | Main CI - builds, tests, Flow checks |
| `shared_lint.yml` | Push to main, PRs | Prettier, ESLint, license checks |
| `compiler_typescript.yml` | PRs touching compiler | Compiler-specific tests |
| `devtools_regression_tests.yml` | PRs | DevTools testing |

### Test Matrix Scale
- **90 test shards** (18 configurations × 5 shards each)
- **50 build jobs** (25 workers × 2 release channels)
- **50 test-build shards** (5 configurations × 10 shards)
- Flow checks across multiple inline configs

---

## 2. Recent Main Branch Status

| Commit | Date | Description | Status |
|--------|------|-------------|--------|
| `bf1afad` | Dec 4, 2025 | [react-dom/server] Fix hanging on Deno | ✅ Merged |
| `0526c79` | Dec 3, 2025 | Update changelog with latest releases | ✅ Merged |
| `7dc903c` | Dec 3, 2025 | Patch FlightReplyServer (security fix) | ✅ Merged |
| `36df5e8` | Dec 2, 2025 | Allow building single release channel | ✅ Merged |

**Last 10 commits:** All successfully merged to main, indicating CI is passing.

---

## 3. Active PR CI Status

| PR | Title | CodeSandbox Status |
|----|-------|-------------------|
| #35267 | Fix spelling (behaviour → behavior) | 🟡 Pending (building) |
| #35238 | DevTools navigating commits hotkey | ✅ Success |
| #35287 | Compiler: Fix variable name issue | ✅ Success |
| #35278 | Add DevTools console suppress option | ✅ Success |
| #35226 | Fizz: Push stalled use() to ownerStack | ✅ Success |

---

## 4. Risk Assessment

### ✅ Positive Indicators
- **Main branch stable**: All recent commits merged successfully
- **No open CI failure issues**: Search returned zero CI-related open bugs
- **Active development**: Security patches and features landing regularly
- **PR builds passing**: Most open PRs show successful builds

### ⚠️ Areas to Monitor
- **Large test matrix**: 190+ parallel jobs mean potential for infrastructure flakiness
- **Playwright-based e2e tests**: Browser-based tests can be flaky (Flight fixtures, DevTools e2e)
- **Cache dependencies**: Multiple cache strategies (v6 keys) - cache misses could slow builds

### 📊 CI Complexity Metrics
- ~37KB workflow file for main CI (`runtime_build_and_test.yml`)
- Heavy parallelization with matrix strategies
- Multiple artifact upload/download operations

---

## 5. Recommended Actions

| Priority | Action | Rationale |
|----------|--------|-----------|
| **LOW** | Monitor PR #35267 | Currently building - verify completion |
| **LOW** | No immediate action required | Main branch healthy, PRs passing |
| **INFO** | Security patch merged Dec 3 | PR #35277 fixed critical security vuln in FlightReplyServer - verify downstream impact |

---

## 6. On-Call Notes

**TL;DR for On-Call Engineer:**
- 🟢 **CI is GREEN** - No action required
- Main branch is healthy with successful merges in last 24h
- All checked PRs showing green/passing status
- No open issues flagged for CI failures or flakiness
- Recent security patch (#35277) was successfully merged - monitor for any regressions

**If issues arise:**
1. Check GitHub Actions tab directly: `https://github.com/facebook/react/actions`
2. Key workflows to monitor: "(Runtime) Build and Test", "(Shared) Lint"
3. Caches use `v6` key prefix - if widespread failures, consider cache invalidation

---

*Analysis performed: December 4, 2025*
*Data sources: GitHub API (commits, PRs, status checks, workflow files)*

This is the core observability loop. Given a loose prompt ("analyze CI health") and a locked-down GitHub tool surface, the agent picked its own investigation path — commits, PRs, issues, workflow runs — and returned a summary structured the way an on-call engineer needs it. The structure comes from the prompt, not from custom tool-orchestration code we had to write.


In [10]:
reset_activity_context()
visualize_conversation(messages)

In [11]:
reset_activity_context()
display_agent_response(messages)

### Observability Agent as Module

The `observability_agent/agent.py` module wraps the observability pattern into a reusable `send_query` function. It imports and uses the shared visualization utilities from `utils.agent_visualizer` internally:
- **`reset_activity_context()`**: Called automatically at the start of each query
- **`print_activity()`**: Provides real-time feedback during execution
- **`display_agent_response()`**: Renders the final result (controlled by `display_result` parameter)

This means you can use the module with minimal code:

In [12]:
# Reload the module to pick up any changes (useful during development)
from observability_agent.agent import send_query

# The module handles activity display, context reset, and result visualization internally
result = await send_query(
    "Check the CI status for the last 2 runs in anthropics/claude-agent-sdk-python. Just do 3 tool calls, be efficient."
)

🤖 Using: mcp__github__list_commits()
✓ Tool completed
🤖 Using: mcp__github__get_commit()
🤖 Using: mcp__github__get_commit()
✓ Tool completed
✓ Tool completed
🤖 Thinking...


Commit,Message,Date,Status
2437035,chore: bump bundled CLI version to 2.0.58,"Dec 3, 2025 20:09 UTC",⚠️ No CI status available
9809fb6,chore: release v0.1.11 (#383),"Dec 3, 2025 19:42 UTC",⚠️ No CI status available


Multi-turn conversations work seamlessly - just pass `continue_conversation=True`:

In [13]:
# Example 2: Multi-turn conversation for deeper monitoring
result1 = await send_query("What's the current CI status for facebook/react?")

🤖 Using: mcp__github__list_pull_requests()
🤖 Using: mcp__github__list_commits()
✓ Tool completed
✓ Tool completed
🤖 Thinking...
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__pull_request_read()
🤖 Using: mcp__github__get_commit()
✓ Tool completed
✓ Tool completed
✓ Tool completed
✓ Tool completed
🤖 Thinking...


PR,Title,Author,CI Status,Updated
#35287,[compiler] Fix JSX variable name issue,@kostya-gromov,🟢 Success,2h ago
#35285,[compiler][poc] Reuse ValidateExhaustiveDeps,@josephsavona,🔵 Draft,6h ago
#35284,[compiler] Fix hoisted primitives bug,@josephsavona,🟢 Success,7h ago
#35282,[compiler] Add effect deps validator,@jackpope,🟢 Success,15h ago
#35281,Improve legacy context warning,@Harshrj53,🟢 Success,20h ago


In [14]:
# Continue the conversation to dig deeper
result2 = await send_query(
    "Are there any flaky tests in the recent failures? You can only make one tool call.",
    continue_conversation=True,
)

🤖 Using: mcp__github__search_issues()
✓ Tool completed
🤖 Thinking...


Metric,Status
Open flaky test issues,0
Recent CI failures,None detected
Test stability,✅ Stable


## Limitations & Considerations

- **Docker cold-start cost.** Every query in this notebook spawns a fresh `ghcr.io/github/github-mcp-server` container. That's fine for interactive exploration and scheduled checks; for high-frequency use, run the MCP server as a long-lived process instead.
- **GitHub API rate limits.** A single CI-triage prompt can fan out into 10–20 tool calls. Fine-grained tokens and the GraphQL API give you higher limits than the legacy REST token flow.
- **Token scope hygiene.** Give `GITHUB_TOKEN` the minimum scope your workflows need. For anything public, the default fine-grained scope is enough — avoid granting `repo` write or admin scopes unless you actually want the agent to mutate state.
- **Read-only by design.** We kept `allowed_tools=["mcp__github"]` broad but the prompt and system prompt constrain the agent to analysis. Before enabling write operations (`create_pull_request`, `issue_write`, `merge_pull_request`), narrow `allowed_tools` to the specific MCP tools you trust and consider a human-in-the-loop `permission_mode`.
- **`allowed_tools` vs `disallowed_tools`.** `allowed_tools` alone only affects permission prompts; `disallowed_tools` is what actually removes a tool from the agent's reach. If you only set `allowed_tools=["mcp__github"]`, the agent can still call `Bash` and reach for `gh` — that's why we pair them.


## Conclusion

### Recap

We met the four objectives set at the top of the notebook:

1. **Configured stdio MCP servers** — Git (local `uv run python -m mcp_server_git`) and GitHub (`ghcr.io/github/github-mcp-server` in Docker).
2. **Locked the agent to an MCP-only tool surface** — `allowed_tools=["mcp__<server>"]` paired with `disallowed_tools=["Bash", "Task", "WebSearch", "WebFetch"]`.
3. **Ran a real triage prompt** against `facebook/react` and got an on-call-ready CI summary.
4. **Wrapped the pattern in a reusable module** — `observability_agent/agent.py::send_query`, usable in a single line from any other code.

### Apply this to your own systems

- **Point it at your repos.** Swap `facebook/react` for one of yours and run the triage prompt on a schedule — cron, a GitHub Actions workflow, or a serverless function.
- **Chain more MCP servers.** Add a Slack or PagerDuty MCP server to post summaries to the right channel, or an internal metrics MCP to correlate CI failures with deploys.
- **Tune the system prompt.** `observability_agent/agent.py` defines `DEFAULT_SYSTEM_PROMPT`. That's the right place to encode severity rubrics, SLAs, and team-specific runbooks.
- **Graduate from read-only.** Once you trust the triage output, narrow `allowed_tools` to specific write tools (e.g. `mcp__github__add_issue_comment`) and let the agent post its findings directly where your team already operates.

### What You've Learned Across All Notebooks

**From Notebook 00 (Research Agent)**
- Core SDK fundamentals with `query()` and `ClaudeSDKClient`
- Basic tool usage with WebSearch and Read
- Simple agent loops and conversation management

**From Notebook 01 (Chief of Staff)**
- Advanced features: memory, output styles, planning mode
- Multi-agent coordination through subagents
- Governance through hooks and custom commands
- Enterprise-ready agent architectures

**From Notebook 02 (Observability Agent)**
- External system integration via MCP servers
- Real-time monitoring and incident response
- Production workflow automation
- Scalable agent deployment patterns

The complete implementations for all three agents live in their respective directories (`research_agent/`, `chief_of_staff_agent/`, `observability_agent/`), ready to adapt into your production systems.
